# Resource usage benchmark
Collects wall time, CPU core-hours, and peak memory for all simulation methods
across three representative molecule sizes (small / medium / large).

## Strategy
| Method | Data source | Notes |
|---|---|---|
| QCxMS GS-MD | `sacct` + SLURM `.out` files | Per-variant, per-molecule |
| QCxMS frag | `sacct` + SLURM `.out` files | Per-variant, per-molecule |
| CREST | `crest.log` ("Total run time") + `sacct` | Reported separately from QCxMS2 |
| QCxMS2 | `qcxms2.log` + `sacct` | Reported separately from CREST |
| NEIMS | SLURM `.out` (`time` wrapper in script) + `sacct` | Fast; re-run optional |
| CFMID | SLURM `.out` + `sacct` | Whole-dataset job; per-mol = total / N |

**Do not re-run** QCxMS / CREST / QCxMS2 — weeks of compute.
NEIMS and CFMID are fast enough to re-run clean for 3 molecules (section 5).

In [ ]:
import os, sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath(".."))

from src.utils.parse_resource_usage import (
    collect_qcxms_resources,
    collect_crest_resources,
    collect_qcxms2_resources,
    collect_neims_resources,
    collect_cfmid_resources,
    select_size_representatives_multi,
    heavy_atom_count,
    fmt_wall, fmt_mem,
    load_sacct_cache, save_sacct_cache,
)

In [ ]:
# ==========================================================================
# CONFIG -- edit only this cell
# ==========================================================================

# Dataset to benchmark (use the one with most complete runs)
DATASET_NAME   = "franklin_tms"
INPUT_CSV      = "../data/processed/franklin_tms/franklin_tms.csv"
SMILES_COLUMN  = "Modified_SMILES"

# QCxMS variants present in this dataset
QCXMS_VARIANTS = ["QCxMS_10_ps", "QCxMS_25_ps", "QCxMS_10_ps_iee03"]

# SLURM username for sacct queries
SLURM_USER = "hsandstr"

# How far back to search sacct (YYYY-MM-DD)
SACCT_START = "2024-01-01"

# Total number of molecules in dataset (for CFMID per-mol normalisation)
N_MOLS = 61

# --------------------------------------------------------------------------
# Memory benchmark: set USE_BENCH_DIR = True to collect MaxRSS from the
# isolated memory_benchmark/ runs (clean per-molecule data).
# When False, collection falls back to the full production dataset.
# --------------------------------------------------------------------------
USE_BENCH_DIR = True

# --------------------------------------------------------------------------
# Benchmark molecule IDs grouped by heavy-atom size.
# These are the 9 molecules submitted to memory_benchmark/ manually.
# Set MOL_IDS = None to auto-select from the full dataset instead.
# --------------------------------------------------------------------------
N_PER_SIZE = 3
MOL_IDS = {
    "small":  ["0038", "0060", "0015"],   # 12, 16, 18 HA
    "medium": ["0028", "0000", "0012"],   # 19, 20, 22 HA
    "large":  ["0033", "0036", "0030"],   # 25, 27, 31 HA
}


## 1. Select representative molecules

In [ ]:
SIM_BASE  = os.path.abspath(f"../data/simulation_results/{DATASET_NAME}")
BENCH_DIR = os.path.abspath(f"../data/simulation_results/{DATASET_NAME}/memory_benchmark")
DATA_DIR  = BENCH_DIR if USE_BENCH_DIR else SIM_BASE
print(f"Data directory: {DATA_DIR}")


def avg_dicts(dicts, keys=('wall_s', 'cpu_s', 'peak_mb')):
    """Average resource dicts, ignoring zero/missing values."""
    if not dicts:
        return {}
    result = {}
    for k in keys:
        vals = [d[k] for d in dicts if d.get(k, 0) > 0]
        if vals:
            result[k] = sum(vals) / len(vals)
    return result

if MOL_IDS is None:
    reps = select_size_representatives_multi(INPUT_CSV, n=N_PER_SIZE, smiles_col=SMILES_COLUMN)
else:
    # MOL_IDS = {"small": ["0003", "0010"], ...}
    df_csv = pd.read_csv(INPUT_CSV).reset_index()
    reps = {}
    for size, mol_ids in MOL_IDS.items():
        reps[size] = []
        for mol_id in mol_ids:
            idx    = int(mol_id)
            row    = df_csv.iloc[idx]
            smiles = row.get(SMILES_COLUMN, "")
            reps[size].append({"mol_id": mol_id, "smiles": smiles,
                                "heavy_atoms": heavy_atom_count(str(smiles))})

SIZES = ("small", "medium", "large")

print("Selected molecules:")
print(f"{'Size':<8} {'ID':<6} {'HA':>4}  SMILES (truncated)")
print("-" * 70)
for size in SIZES:
    for r in reps[size]:
        print(f"{size:<8} {r['mol_id']:<6} {r['heavy_atoms']:>4}  {str(r['smiles'])[:45]}")

In [ ]:
# Load persisted sacct records so MaxRSS is available even after SLURM
# accounting records expire. Run section 5c to populate this cache.
cache_path = f"{BENCH_DIR}/sacct_cache.json"
if os.path.exists(cache_path):
    load_sacct_cache(cache_path)
else:
    print(f"No sacct cache found at {cache_path} — will query live sacct.")

## 2. Collect resource data from existing runs

Reads SLURM accounting (`sacct`) and log files.
Each cell collects data for one method group and can be re-run independently.

In [ ]:
# -- QCxMS (all variants, GS-MD + fragmentation) ----------------------------
qcxms_data = {}   # {variant: {size: {"gsmd": avg_dict, "frag": avg_dict}}}

for variant in QCXMS_VARIANTS:
    qcxms_data[variant] = {}
    for size in SIZES:
        gsmd_results, frag_results = [], []
        for rep in reps[size]:
            mol_id = rep["mol_id"]
            print(f"  {variant} / {size} ({mol_id}) ...")
            r = collect_qcxms_resources(DATA_DIR, mol_id, variant, user=SLURM_USER)
            gs, fr = r["gsmd"], r["frag"]
            print(f"    GS-MD : wall={fmt_wall(gs.get('wall_s',0))}  mem={fmt_mem(gs.get('peak_mb',0))}")
            print(f"    Frag  : wall={fmt_wall(fr.get('wall_s',0))}  mem={fmt_mem(fr.get('peak_mb',0))}")
            if gs.get('wall_s', 0): gsmd_results.append(gs)
            if fr.get('wall_s', 0): frag_results.append(fr)
        qcxms_data[variant][size] = {
            "gsmd": avg_dicts(gsmd_results),
            "frag": avg_dicts(frag_results),
        }
        print(f"  >> {size} avg: GS-MD={fmt_wall(qcxms_data[variant][size]['gsmd'].get('wall_s',0))}  "
              f"Frag={fmt_wall(qcxms_data[variant][size]['frag'].get('wall_s',0))}")

In [ ]:
# -- CREST ------------------------------------------------------------------
crest_data = {}   # {size: avg_dict}

for size in SIZES:
    results = []
    for rep in reps[size]:
        mol_id = rep["mol_id"]
        print(f"  CREST / {size} ({mol_id}) ...")
        r = collect_crest_resources(DATA_DIR, mol_id, user=SLURM_USER)
        src = r.get('source', 'not found')
        print(f"    wall={fmt_wall(r.get('wall_s',0))}  mem={fmt_mem(r.get('peak_mb',0))}  src={src}")
        if r.get('wall_s', 0): results.append(r)
    crest_data[size] = avg_dicts(results)
    print(f"  >> {size} avg: wall={fmt_wall(crest_data[size].get('wall_s',0))}")

In [ ]:
# -- QCxMS2 -----------------------------------------------------------------
qcxms2_data = {}   # {size: avg_dict}

for size in SIZES:
    results = []
    for rep in reps[size]:
        mol_id = rep["mol_id"]
        print(f"  QCxMS2 / {size} ({mol_id}) ...")
        r = collect_qcxms2_resources(DATA_DIR, mol_id, user=SLURM_USER)
        src = r.get('source', 'not found')
        print(f"    wall={fmt_wall(r.get('wall_s',0))}  mem={fmt_mem(r.get('peak_mb',0))}  src={src}")
        if r.get('wall_s', 0): results.append(r)
    qcxms2_data[size] = avg_dicts(results)
    print(f"  >> {size} avg: wall={fmt_wall(qcxms2_data[size].get('wall_s',0))}")

In [ ]:
# -- NEIMS ------------------------------------------------------------------
neims_data = {}   # {size: avg_dict}

for size in SIZES:
    results = []
    for rep in reps[size]:
        mol_id = rep["mol_id"]
        print(f"  NEIMS / {size} ({mol_id}) ...")
        r = collect_neims_resources(DATA_DIR, mol_id, user=SLURM_USER)
        src = r.get('source', 'not found')
        print(f"    wall={fmt_wall(r.get('wall_s',0))}  mem={fmt_mem(r.get('peak_mb',0))}  src={src}")
        if r.get('wall_s', 0): results.append(r)
    neims_data[size] = avg_dicts(results)
    print(f"  >> {size} avg: wall={fmt_wall(neims_data[size].get('wall_s',0))}")

In [ ]:
# -- CFMID (whole-dataset job; divide by N_MOLS for per-molecule estimate) --
print("  CFMID (whole-dataset job) ...")
cfmid_total = collect_cfmid_resources(DATA_DIR, user=SLURM_USER)

cfmid_per_mol = {
    "wall_s":  cfmid_total.get("wall_s",  0) / N_MOLS,
    "peak_mb": cfmid_total.get("peak_mb", 0),   # memory is shared, report total
    "n_cpus":  cfmid_total.get("n_cpus",  1),
}
print(f"  Total  : wall={fmt_wall(cfmid_total.get('wall_s',0))}  "
      f"mem={fmt_mem(cfmid_total.get('peak_mb',0))}")
print(f"  Per mol (/{N_MOLS}): wall={fmt_wall(cfmid_per_mol['wall_s'])}")

## 3. Summary table

In [ ]:
SIZES     = ("small", "medium", "large")
SIZE_COLS = {
    s: f"{s.capitalize()} ({min(r['heavy_atoms'] for r in reps[s])}–"
       f"{max(r['heavy_atoms'] for r in reps[s])} HA)"
    for s in SIZES
}

# SLURM requested resources (from the submission scripts -- fixed, not measured)
REQUESTED = {
    "QCxMS GS-MD": {"cpus": 1,  "mem_gb": 128},
    "QCxMS frag":  {"cpus": 1,  "mem_gb": 8},
    "CREST":       {"cpus": 8,  "mem_gb": 32},
    "QCxMS2":      {"cpus": 32, "mem_gb": 64},
    "NEIMS":       {"cpus": 1,  "mem_gb": 4},
    "CFMID":       {"cpus": 1,  "mem_gb": None},
}

def make_cell(wall_s, cpu_s, peak_mb, n_cpus):
    if not wall_s:
        return "-"
    core_h = cpu_s / 3600 if cpu_s else wall_s * n_cpus / 3600
    parts  = [fmt_wall(wall_s)]
    if core_h:
        parts.append(f"{core_h:.2f} core-h")
    if peak_mb:
        parts.append(fmt_mem(peak_mb))
    return " / ".join(parts)

rows = []

# QCxMS rows
for variant in QCXMS_VARIANTS:
    vname = (variant.replace("QCxMS_", "").replace("_ps", " ps")
                    .replace("_iee03", " iee=0.3 eV"))
    for step in ("gsmd", "frag"):
        req_key = "QCxMS GS-MD" if step == "gsmd" else "QCxMS frag"
        req     = REQUESTED[req_key]
        row = {
            "Method":  f"QCxMS ({vname})",
            "Step":    "GS-MD" if step == "gsmd" else "Fragmentation",
            "CPUs":    req["cpus"],
            "Mem req": f"{req['mem_gb']} GB",
            "GPU":     "-",
        }
        for size in SIZES:
            d = qcxms_data.get(variant, {}).get(size, {}).get(step, {})
            row[SIZE_COLS[size]] = make_cell(
                d.get("wall_s", 0), d.get("cpu_s", 0),
                d.get("peak_mb", 0), req["cpus"]
            )
        rows.append(row)

# CREST
req = REQUESTED["CREST"]
row = {"Method": "CREST (for QCxMS2)", "Step": "-",
       "CPUs": req["cpus"], "Mem req": f"{req['mem_gb']} GB", "GPU": "-"}
for size in SIZES:
    d = crest_data.get(size, {})
    row[SIZE_COLS[size]] = make_cell(
        d.get("wall_s",0), d.get("cpu_s",0), d.get("peak_mb",0), req["cpus"])
rows.append(row)

# QCxMS2
req = REQUESTED["QCxMS2"]
row = {"Method": "QCxMS2 (GFN2)", "Step": "-",
       "CPUs": req["cpus"], "Mem req": f"{req['mem_gb']} GB", "GPU": "-"}
for size in SIZES:
    d = qcxms2_data.get(size, {})
    row[SIZE_COLS[size]] = make_cell(
        d.get("wall_s",0), d.get("cpu_s",0), d.get("peak_mb",0), req["cpus"])
rows.append(row)

# NEIMS
req = REQUESTED["NEIMS"]
row = {"Method": "NEIMS", "Step": "-",
       "CPUs": req["cpus"], "Mem req": f"{req['mem_gb']} GB", "GPU": "CPU only"}
for size in SIZES:
    d = neims_data.get(size, {})
    row[SIZE_COLS[size]] = make_cell(
        d.get("wall_s",0), d.get("cpu_s",0), d.get("peak_mb",0), req["cpus"])
rows.append(row)

# CFMID
row = {"Method": "CFM-ID", "Step": "-", "CPUs": 1, "Mem req": "-", "GPU": "-"}
for size in SIZES:
    row[SIZE_COLS[size]] = fmt_wall(cfmid_per_mol["wall_s"]) + " (est.)"
rows.append(row)

df_table = pd.DataFrame(rows).set_index(["Method", "Step"])
pd.set_option("display.max_colwidth", 50)
display(df_table)

## 4. Scaling visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

PALETTE = {
    "QCxMS (10 ps) GS-MD":          "#5A99D3",
    "QCxMS (10 ps) Fragmentation":   "#a8c8e8",
    "QCxMS (25 ps) GS-MD":          "#D36EA5",
    "QCxMS (25 ps) Fragmentation":   "#e8b0cc",
    "QCxMS (10 ps iee=0.3 eV) GS-MD":          "#7B68EE",
    "QCxMS (10 ps iee=0.3 eV) Fragmentation":   "#b8b0f0",
    "CREST (for QCxMS2)":            "#F0A500",
    "QCxMS2 (GFN2)":                 "#E07B39",
    "NEIMS":                         "#4C4C4C",
    "CFM-ID":                        "#A6A6A6",
}

plot_rows = []
for variant in QCXMS_VARIANTS:
    vname = (variant.replace("QCxMS_", "").replace("_ps", " ps")
                    .replace("_iee03", " iee=0.3 eV"))
    for step in ("gsmd", "frag"):
        label = f"QCxMS ({vname}) {'GS-MD' if step=='gsmd' else 'Fragmentation'}"
        for size in SIZES:
            d = qcxms_data.get(variant, {}).get(size, {}).get(step, {})
            plot_rows.append({"method": label, "size": size,
                              "ha": sum(r["heavy_atoms"] for r in reps[size]) / len(reps[size]),
                              "wall_min": d.get("wall_s", 0) / 60})

for label, data_dict in [("CREST (for QCxMS2)", crest_data),
                          ("QCxMS2 (GFN2)",       qcxms2_data),
                          ("NEIMS",               neims_data)]:
    for size in SIZES:
        d = data_dict.get(size, {})
        plot_rows.append({"method": label, "size": size,
                          "ha": sum(r["heavy_atoms"] for r in reps[size]) / len(reps[size]),
                          "wall_min": d.get("wall_s", 0) / 60})

for size in SIZES:
    plot_rows.append({"method": "CFM-ID", "size": size,
                      "ha": sum(r["heavy_atoms"] for r in reps[size]) / len(reps[size]),
                      "wall_min": cfmid_per_mol["wall_s"] / 60})

df_plot = pd.DataFrame(plot_rows)

fig, ax = plt.subplots(figsize=(9, 5))
for method, grp in df_plot.groupby("method"):
    grp = grp.sort_values("ha")
    has_data = grp["wall_min"] > 0
    if has_data.any():
        color = PALETTE.get(method, "#888888")
        ax.plot(grp.loc[has_data, "ha"], grp.loc[has_data, "wall_min"],
                "o-", label=method, color=color,
                linewidth=1.8, markersize=7, markeredgewidth=0)

ax.set_xlabel("Heavy atom count", fontsize=13)
ax.set_ylabel("Wall time (min)", fontsize=13)
ax.set_title("Computational cost vs. molecule size", fontsize=14, weight="bold")
ax.set_yscale("log")
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda x, _: f"{x:.0f}" if x >= 1 else f"{x:.2f}")
)
ax.legend(fontsize=8, ncol=2, frameon=True, framealpha=0.9,
          bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/resource_scaling.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Optional: clean benchmark re-run for fast methods (NEIMS / CFMID)

NEIMS takes ~10 s per molecule — re-running the three benchmark molecules
in isolation gives cleaner timing than parsing shared-queue jobs.

Set `RUN_BENCHMARK = True` and re-run this cell to submit.

In [ ]:
RUN_BENCHMARK = False   # set True to submit

if RUN_BENCHMARK:
    from src.workflow.job_submission import submit_slurm_array
    SRC_ROOT  = os.path.abspath("../src")
    NEIMS_DIR = f"{SIM_BASE}/NEIMS"
    task_ids  = ",".join(
        str(int(rep["mol_id"])) for size in SIZES for rep in reps[size]
    )
    print(f"Submitting NEIMS for molecules: {task_ids}")
    submit_slurm_array(
        sim_dir     = NEIMS_DIR,
        script_path = f"{SRC_ROOT}/workflow/submit_neims_array.sh",
        array_spec  = task_ids,
    )
    print("Re-run section 2 cells after jobs complete to collect fresh timings.")
else:
    print("RUN_BENCHMARK=False -- skipping.")

In [ ]:
RUN_CFMID_BENCHMARK = False  # set True to resubmit

if RUN_CFMID_BENCHMARK:
    import subprocess
    CFMID_DIR  = os.path.abspath(f"{SIM_BASE}/CFMID")
    SRC_ROOT   = os.path.abspath("../src")
    script     = f"{SRC_ROOT}/workflow/run_cfmid.sh"
    result = subprocess.run(
        ["sbatch", script, CFMID_DIR],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True,
        cwd=CFMID_DIR,
    )
    print(result.stdout.strip() or result.stderr.strip())
    print("Re-run section 2 (cell 10) after the job completes to collect fresh timing.")
else:
    print("RUN_CFMID_BENCHMARK=False -- skipping.")

In [ ]:
# ==========================================================================
# MEMORY BENCHMARK — submit isolated runs for clean MaxRSS measurements
# ==========================================================================
# Workflow:
#   1. Set RUN_MEMORY_BENCHMARK=True → submits NEIMS, CFMID, QCxMS GS-MD
#   2. After GS-MD finishes, submit fragmentation (see printed instructions)
#   3. Set RUN_SLOW_METHODS=True    → submits CREST for each benchmark molecule
#   4. After CREST finishes, run section 5b cell to submit QCxMS2
#   5. After all jobs finish, run section 5c to save the sacct cache
# ==========================================================================

from src.workflow.job_submission import (
    setup_memory_benchmark, submit_crest_for_benchmark,
)

RUN_MEMORY_BENCHMARK = False   # NEIMS + CFMID + QCxMS GS-MD
RUN_SLOW_METHODS     = False   # CREST (prerequisite for QCxMS2)

SRC_ROOT      = os.path.abspath("../src")
CREST_SCRIPT  = os.path.abspath(f"{SRC_ROOT}/workflow/submit_batch_crest.sh")
QCXMS2_SCRIPT = os.path.abspath(f"{SRC_ROOT}/workflow/submit_qcxms2_job.sh")
bench_mols    = [rep for size in SIZES for rep in reps[size]]

if RUN_MEMORY_BENCHMARK:
    job_ids = setup_memory_benchmark(
        bench_dir      = BENCH_DIR,
        sim_base       = SIM_BASE,
        bench_mols     = bench_mols,
        qcxms_variants = QCXMS_VARIANTS,
        src_root       = SRC_ROOT,
    )
    if job_ids:
        save_sacct_cache(",".join(job_ids), f"{BENCH_DIR}/sacct_cache.json")
else:
    print("RUN_MEMORY_BENCHMARK=False — skipping fast-method setup.")

if RUN_SLOW_METHODS:
    job_ids = submit_crest_for_benchmark(
        bench_dir    = BENCH_DIR,
        bench_mols   = bench_mols,
        crest_script = CREST_SCRIPT,
    )
    if job_ids:
        save_sacct_cache(",".join(job_ids), f"{BENCH_DIR}/sacct_cache.json")
else:
    print("RUN_SLOW_METHODS=False — skipping CREST submission.")

## 5b. Targeted resubmission — CREST and QCxMS2 for missing benchmark molecules

Checks each benchmark molecule for `peaks.csv` (QCxMS2 done) and submits
whatever is still needed:
- No `crest_best.xyz` → submit CREST
- `crest_best.xyz` present but no `peaks.csv` → submit QCxMS2
  (script auto-detects `getieeab` error and retries with `-edist gaussian`)

Set `SUBMIT_MISSING = True` to actually submit jobs.

In [ ]:
from src.workflow.job_submission import check_and_submit_qcxms2

SUBMIT_MISSING = True   # set True to submit

check_and_submit_qcxms2(
    bench_dir     = BENCH_DIR,
    reps          = reps,
    sizes         = SIZES,
    crest_script  = CREST_SCRIPT,
    qcxms2_script = QCXMS2_SCRIPT,
    submit        = SUBMIT_MISSING,
)

## 5c. Save sacct cache after all benchmark jobs complete

Run this cell once every benchmark job has finished (`squeue -u $USER` shows nothing
from the benchmark).  Sacct step records (which hold MaxRSS) are kept for ~1 year;
saving them now means the resource table can still be regenerated after they expire.

After saving, re-run the section 2 collection cells to pick up the fresh MaxRSS values.

In [ ]:
# Paste the comma-separated SLURM job IDs for all completed benchmark jobs.
# Find them via: sacct -u $USER --starttime=2025-01-01 --format=JobID,JobName,State
BENCH_JOB_IDS = ""   # e.g. "34052083,34052084,34052085,34052086"

if BENCH_JOB_IDS:
    save_sacct_cache(BENCH_JOB_IDS, f"{BENCH_DIR}/sacct_cache.json")
    load_sacct_cache(f"{BENCH_DIR}/sacct_cache.json")
    print("Cache saved. Re-run section 2 cells to update resource data.")
else:
    print("Set BENCH_JOB_IDS, then re-run this cell.")

## 6. LaTeX table for paper

In [ ]:
CAPTION = (
    "Computational resource requirements per molecule for each simulation method, "
    "evaluated on the Franklin TMS dataset (CSC Puhti cluster). "
    "Wall time and peak memory for small, medium, and large molecules (by heavy-atom count). "
    "CREST and QCxMS2 are reported separately. "
    "CFM-ID wall time is estimated as total job wall time divided by number of molecules. "
    "NEIMS uses CPU only."
)
LABEL = "tab:resource_usage"

lines = []
lines.append(r"\begin{table}[htbp]")
lines.append(r"\centering")
lines.append(r"\small")
lines.append(r"\caption{" + CAPTION + "}")
lines.append(r"\label{" + LABEL + "}")
lines.append(r"\begin{tabular}{ll rr " + "rr " * len(SIZES) + "}")
lines.append(r"\toprule")

size_header = " & ".join(
    f"\\multicolumn{{2}}{{c}}{{{s.capitalize()} "
    f"({min(r['heavy_atoms'] for r in reps[s])}–"
    f"{max(r['heavy_atoms'] for r in reps[s])} HA)}}"
    for s in SIZES
)
lines.append(f"Method & Step & CPUs & Mem\\textsuperscript{{req}} & {size_header} \\\\")
lines.append("& & & & " + " & ".join(["Wall & Mem"] * len(SIZES)) + " \\\\")
lines.append(r"\midrule")

def lwall(wall_s):
    return fmt_wall(wall_s) if wall_s else "---"

def lmem(peak_mb):
    return fmt_mem(peak_mb) if peak_mb else "---"

for variant in QCXMS_VARIANTS:
    vname = (variant.replace("QCxMS_","").replace("_ps"," ps")
                    .replace("_iee03"," $\\varepsilon$=0.3\\,eV"))
    for step in ("gsmd", "frag"):
        req_key  = "QCxMS GS-MD" if step == "gsmd" else "QCxMS frag"
        req      = REQUESTED[req_key]
        step_str = "GS-MD" if step == "gsmd" else "Frag."
        cells    = " & ".join(
            f"{lwall(qcxms_data.get(v,{}).get(s,{}).get(step,{}).get('wall_s',0))} & "
            f"{lmem(qcxms_data.get(v,{}).get(s,{}).get(step,{}).get('peak_mb',0))}"
            for v, s in [(variant, sz) for sz in SIZES]
        )
        m_str = f"QCxMS ({vname})" if step == "gsmd" else ""
        lines.append(f"{m_str} & {step_str} & {req['cpus']} & {req['mem_gb']}\\,GB & {cells} \\\\")
    lines.append(r"\addlinespace")

req   = REQUESTED["CREST"]
cells = " & ".join(
    f"{lwall(crest_data.get(s,{}).get('wall_s',0))} & "
    f"{lmem(crest_data.get(s,{}).get('peak_mb',0))}" for s in SIZES
)
lines.append(f"CREST & --- & {req['cpus']} & {req['mem_gb']}\\,GB & {cells} \\\\")

req   = REQUESTED["QCxMS2"]
cells = " & ".join(
    f"{lwall(qcxms2_data.get(s,{}).get('wall_s',0))} & "
    f"{lmem(qcxms2_data.get(s,{}).get('peak_mb',0))}" for s in SIZES
)
lines.append(f"QCxMS2 (GFN2) & --- & {req['cpus']} & {req['mem_gb']}\\,GB & {cells} \\\\")
lines.append(r"\addlinespace")

req   = REQUESTED["NEIMS"]
cells = " & ".join(
    f"{lwall(neims_data.get(s,{}).get('wall_s',0))} & "
    f"{lmem(neims_data.get(s,{}).get('peak_mb',0))}" for s in SIZES
)
lines.append(f"NEIMS & --- & {req['cpus']} & {req['mem_gb']}\\,GB & {cells} \\\\")

cells = " & ".join(f"{lwall(cfmid_per_mol['wall_s'])} (est.) & ---" for _ in SIZES)
lines.append(f"CFM-ID & --- & 1 & --- & {cells} \\\\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

tex = "\n".join(lines)
print(tex)

out_dir = Path("../reports/mem_benchmark")
out_dir.mkdir(parents=True, exist_ok=True)
tex_path = out_dir / "resource_usage.tex"
tex_path.write_text(tex)
print(f"\nSaved: {tex_path}")